In [1]:
import numpy as np
import scipy.linalg as la
import scipy.integrate as sint
import sympy as sp
from sympy.functions.special.bsplines import bspline_basis
from sympy import chebyshevt, chebyshevu, sqrt, pi, symbols
from sympy import legendre
import matplotlib.pyplot as plt
import itertools
from scipy.interpolate import interp1d

In [3]:
# Chebyshev points on interval [-1,1] 
def chebyshev_points(N):
    return np.cos(np.pi * np.arange(N) / (N - 1))

# Maxwellian M(x,z)
def Maxwellian(rho_x, T_x, x_p, z_q):
    d_v = 1
    rho_vals = rho_x(x_p)[:, None]                  # shape (N, 1)
    T_vals = T_x(x_p)[:, None]                      # shape (N, 1)
    z_vals = z_q[None, :]                           # shape (1, N)
    return (rho_vals / (2 * np.pi * T_vals)**(d_v / 2) )* np.exp(-z_vals**2 / 2)

# Maxwellian M(x,v)
def Maxwellian_v(rho_x, T_x, u_x, x_p, v_q):
    d_v = 1
    rho_vals = rho_x(x_p)[:, None]                  # shape (N, 1)
    T_vals = T_x(x_p)[:, None]                      # shape (N, 1)
    v_vals = v_q[None, :] # shape (1, N)
    u_vals = u_x(x_p)[None, :] # shape (1, N)
    return (rho_vals / (2 * np.pi * T_vals)**(d_v / 2)) * np.exp(-(v_vals - u_vals)**2 / (2 * T_vals.T))

def Maxwellian_vals(rho_x, T_x,u_x,T_new,u_new, x_p, z_q):
    d_v = 1
    rho_vals = rho_x(x_p)
    T_vals = T_x(x_p)
    z_vals = z_q * np.sqrt(T_new(x_p)) + u_new(x_p) - u_x(x_p)
    return (rho_vals / (2 * np.pi * T_vals)**(d_v / 2) )* np.exp(-z_vals**2 / 2)
# Spatial basis functions 
def phi(n, x):
    Tn = chebyshevt(n, x)
    if n == 0: #orthonormalization
        return Tn / np.sqrt(np.pi)
    else:
        return Tn * np.sqrt(2/np.pi)  

# Derivative of spatial basis functions
def dphi(n, x):
    if n == 0:
        return 0
    else:
        Un_1 = chebyshevu(n-1, x)  
        dTn = n * Un_1
        return dTn * np.sqrt(2/np.pi)  

# Velocity basis functions 
def psi(n, z):
    Tn = chebyshevt(n, z)
    if n == 0: #orthonormalization
        return Tn / np.sqrt(np.pi)
    else:
        return Tn * np.sqrt(2/np.pi)

# Derivative of velocity basis functions
def dpsi(n, z):
    if n == 0:
        return 0  
    else:
        Un_1 = chebyshevu(n-1, z)  
        dTn = n * Un_1
        return dTn * np.sqrt(2/np.pi)     

# Compute velocity variables v = z sqrt(T(x)) + u(x)
def v(T_x, u_x, x_p, z_q):
    sqrt_T = np.sqrt(T_x(x_p))[:, None]    # shape (N, 1)
    u_val = u_x(x_p)[:, None]              # shape (N, 1)
    z = z_q[None, :]                       # shape (1, N)
    return z * sqrt_T + u_val              # shape (N, N)

# Initial f
def f_ini_x(x):
    return np.cos(x)+.00001*x
def f_ini_s(s):
    return np.exp(-(s)**2/2)



# Evaluate f(x,z) at Chebyshev collocation points f(x_p,z_q)
def eval_f(C, A, B): 
    f_val = A.T @ C @ B 
    return f_val
    
# Numeric derivative
def numeric_derivative(f, x, dx=1e-6):
    return (f(x + dx) - f(x - dx)) / (2 * dx)
    
def central_diff(f, dx):
    I1 = np.zeros_like(f)
    I1[1:-1] = -(f[2:] - f[:-2]) / (2 * dx)
    I1[0]    = -(f[1] - f[0]) / dx
    I1[-1]   = -(f[-1] - f[-2]) / dx
    return I1
    
def dz_x(T_x, u_x, x_p, z_q):
    N = len(x_p)
    dzx = np.zeros((N,N))
    for p in range(N):
        dT_dx = numeric_derivative(T_x, x_p[p])
        du_dx = numeric_derivative(u_x, x_p[p])
        Tx_val = T_x(x_p[p])
        for q in range(N):
            # Compute dz/dx element
            dzx[p,q] = (-du_dx / np.sqrt(Tx_val)) - (dT_dx * z_q[q]) / (2 * Tx_val)
    return dzx
    
# Evaluate derivative of f againt x at collocation points D_xf(x_p, z_q)
def eval_D_xf(C, A, B, A_der, B_der, x_p, z_q, T_x, u_x):
    Df_val = A_der.T @ C @ B + np.multiply(dz_x(T_x, u_x, x_p, z_q), A.T @ C @ B_der) 
    return Df_val

# def eval_D_xf_v(C, A, B, A_der, B_der, x_p, z_q, T_x, u_x):
#     Df_val = A_der.T @ C @ B + np.multiply(dz_x(T_x, u_x, x_p, z_q), A.T @ C @ B_der) 
#     return Df_val

def f_check(C,x,z,u,T, u_new, T_new, dt,rho):
    phi_vals = np.array([f(x) for f in phis], dtype=float)
    psi_vals = np.array([f((z * np.sqrt(T_new(x)) + u_new(x) - u(x)) / np.sqrt(T(x))) for f in psis], dtype=float)
    dphi_vals = np.array([f(x) for f in dphis], dtype=float)
    dpsi_vals = np.array([f((z * np.sqrt(T_new(x)) + u_new(x) - u(x)) / np.sqrt(T(x))) for f in dpsis], dtype=float)
    dT_dx = numeric_derivative(T, x)
    du_dx = numeric_derivative(u, x)
    dz_dx = -du_dx /np.sqrt(T(x)) -(z * np.sqrt(T_new(x)) + u_new(x) - u(x)) * dT_dx/ (2*T(x))
    MM = Maxwellian_vals(rho, T,u,T_new,u_new, x, z)
    result = np.sum(C*np.outer(phi_vals,psi_vals)) -dt*(np.multiply((z * np.sqrt(T_new(x)) + u_new(x) - u(x)) * np.sqrt(T(x)) + u(x), np.sum(C*np.outer(dphi_vals,psi_vals)) + np.multiply(np.sum(C*np.outer(phi_vals,dpsi_vals)) , dz_dx)) -nu/epsilon *(MM - np.sum(C*np.outer(phi_vals,psi_vals))))
    return result
    
# def Dfnx_v(fn_v, dx):
#     #dx = x_p[1] - x_p[0] 
#     df_dx = (fn_v[1:, :] - fn_v[:-1, :]) / dx  # shape (N-1, N)
#     last_row = (fn_v[-1, :] - fn_v[-2, :]) / dx
#     df_dx = np.vstack([df_dx, last_row])
#     return df_dx    